##  Задания 4 и 5 

In [ ]:
from math import *
import pandas as pd

#  уравнение: ln(x+5) = cos(x)  следовательно меняем на  f(x) = ln(x+5) - cos(x) = 0
def f(x):
    return log(x + 5) - cos(x)

#  производная
def f_prime(x):
    return 1/(x + 5) + sin(x)

def combined_method_chord_tangent(a, b, tol=1e-6, max_iter=100):
    
    print(f"Интервал: [{a}, {b}]")
    print(f"Точность: {tol}\n")
    
    if f(a) * f(b) >= 0:
        print("Ошибка: на концах интервала функция должна иметь разные знаки.")
        return None
    
    #  определяем, с какой стороны какая формула
    #  для простоты: метод Ньютона с того конца, где значение функции ближе к нулю

    if abs(f(a)) < abs(f(b)):
        #  начальное приближение для Ньютона
        x_newton = a 
        #  начальное приближение для хорд 
        x_chord = b   
    else:
        x_newton = b
        x_chord = a
    
    table_data = []
    
    for n in range(max_iter):
        #  метод касательных (Ньютона)
        if abs(f_prime(x_newton)) < 1e-12:
            print("Производная близка к нулю, метод Ньютона расходится")
            return None
        x_tangent = x_newton - f(x_newton) / f_prime(x_newton)
        
        #  метод хорд (секущих)
        x_chord_new = x_chord - f(x_chord) * (x_chord - x_newton) / (f(x_chord) - f(x_newton))
        
        #  комбинированный подход: берётся среднее арифметическое
        x_next = (x_tangent + x_chord_new) / 2
        
        table_data.append({
            'n': n,
            'x_newton': round(x_newton, 8),
            'x_chord': round(x_chord, 8),
            'x_next': round(x_next, 8),
            'f(x_next)': round(f(x_next), 8),
            '|Δx|': round(abs(x_next - x_newton), 8)
        })
        
        if abs(x_next - x_newton) < tol and abs(x_next - x_chord) < tol:
            break
        
        #  обновляем приближения
        if f(x_next) * f(x_newton) < 0:
            x_chord = x_newton
        else:
            x_chord = x_chord_new
            
        x_newton = x_next
    
    df = pd.DataFrame(table_data)
    print(df.to_string(index=False))
    
    root = x_next
    print(f"\nКорень (комбинированный метод): {root:.8f}")
    print(f"Проверка: ln({root:.8f}+5) = {log(root+5):.8f}")
    print(f"cos({root:.8f}) = {cos(root):.8f}")
    print(f"Невязка: {abs(f(root)):.2e}")
    
    return root

#  интервал для корня x < 5 (из предыдущих заданий: корень около -4.342)
a, b = -4.4, -4.3
combined_method_chord_tangent(a, b, tol=1e-6)

Интервал: [-4.4, -4.3]
Точность: 1e-06

 n  x_newton   x_chord    x_next     f(x_next)         |Δx|
 0 -4.300000 -4.400000 -4.318319  7.562000e-04 1.831898e-02
 1 -4.318319 -4.317820 -4.318635 -2.300000e-07 3.164000e-04
 2 -4.318635 -4.318319 -4.318635  0.000000e+00 9.000000e-08
 3 -4.318635 -4.318635 -4.318635 -0.000000e+00 0.000000e+00

Корень (комбинированный метод): -4.31863528
Проверка: ln(-4.31863528+5) = -0.38365756
cos(-4.31863528) = -0.38365756
Невязка: 9.99e-16


-4.318635283821389

>##  Метод Брента (Brent's method) ##
>Это гибридный (комбинированный) метод поиска корня уравнения f(x)=0, который объединяет три метода: 
>
	>>1. Метод деления отрезка пополам (бисекция) — медленный, но гарантированно сходящийся. 
	>>2. Метод секущих (secant method) — быстрее, но может расходиться.
	>>3. Метод обратной квадратичной интерполяции (inverse quadratic interpolation) — ещё быстрее, но менее надёжен.
>
>>__Идея:__
>
>На каждом шаге Брент пытается применить быстрые методы (секущих или IQI), но если они ведут к нестабильности или выходу за пределы интервала, он "откатывается" к методу деления пополам, который гарантирует, что интервал с корнем будет уменьшаться.
Алгоритм (упрощённо):
>
	>>1. Задаётся начальный интервал [a,b], где f(a)⋅f(b)<0 (знаки разные — корень внутри).
	>>2. На каждой итерации:
>
	    >>>* Пытается применить обратную квадратичную интерполяцию или метод секущих, чтобы получить новое приближение c.
	    >>>* Если c лежит внутри интервала и сходимость "хорошая" — принимает c.
	    >>>* Если нет — делает шаг методом деления пополам: c=(a+b)/2.
>
	>>3. Обновляет интервал: новый [a,b] — это меньший отрезок, содержащий корень.
	>>4. Критерий остановки: ∣b-a∣<"точность"  или ∣f(c)∣<"точность" .


In [ ]:
from scipy import optimize

#  определяем уравнение
def f(x):
    return log(x + 5) - cos(x)

#  ищем корень методом Брента (самый надежный) в интервале [-4.9, -4]
result = optimize.root_scalar(f, bracket=[-4.9, -4], method='brentq')

print(f"Корень найден через SciPy: {result.root:.7f}")
print(f"Количество итераций: {result.iterations}")

Корень найден через SciPy: -4.3186353
Количество итераций: 7
